# Análisis Exploratorio de Datos (EDA)

**Proyecto de Administración Actuarial — FES Acatlán, UNAM · Semestre 2026-1**

Objetivo: entender la base de 292,849 solicitudes de crédito personal, validar la construcción de la variable objetivo Bueno/Malo y verificar que discrimina de forma natural (por `grade`, plazo, etc.).

> El código productivo vive en `src/`. Este notebook documenta y valida el proceso.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import load_data, features

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
raw = load_data.load_raw()
feat = features.build_features(raw)
tgt = features.build_target(raw)
print('Dimensiones:', raw.shape)
print('Columnas:', len(raw.columns))

## 1. Vintages y ventana de observación

Los créditos se emiten entre 2007 y 2015. El corte de información es 2016-01. Los plazos de 36/60 meses implican que los créditos recientes aún están **activos (censurados)** y no pueden catalogarse todavía.

In [ ]:
raw['issue_year'] = raw['issue_d'].dt.year
vintage = raw.groupby('issue_year').size().reset_index(name='n')
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(vintage['issue_year'], vintage['n'], color='#1f77b4')
ax.set_title('Número de créditos por año de emisión')
ax.set_xlabel('Año'); ax.set_ylabel('Créditos');
plt.show()

## 2. Construcción de la variable objetivo

La base no contiene el estatus final del préstamo. Lending Club registra el cargo a pérdida (*charge-off*) escribiendo el saldo principal a cero (`out_prncp == 0`) y marcando después lo recuperado en `recoveries`/`collection_recovery_fee`. Por ello:

- **Resueltos**: `out_prncp == 0` (pagado **o** castigado)
- **Malos**: resueltos con `recoveries > 0` o `collection_recovery_fee > 0`
- **Buenos**: resueltos sin señales de cobranza
- **Censurados**: `out_prncp > 0` (activos, se excluyen de desarrollo)

In [ ]:
resumen = pd.DataFrame({
    'Tipo': ['Resueltos (bueno)', 'Resueltos (malo)', 'Activos (censurado)'],
    'n': [tgt['good'].sum(), tgt['bad'].sum(), tgt['censored'].sum()]
})
resumen['%'] = resumen['n'] / len(tgt) * 100
resumen

In [ ]:
m = feat.merge(tgt[tgt['resolved']], on=['id', 'issue_d'])
tabla = m.groupby('grade').agg(n=('bad','size'), malos=('bad','sum'))
tabla['tasa_mora'] = tabla['malos'] / tabla['n']
tabla

La tasa de mora es **monótona creciente** en la calificación interna A→G: el proxy es consistente con la lógica de negocio del originador.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tabla.index, tabla['tasa_mora'] * 100, 'o-', color='#d62728')
ax.set_title('Tasa de mora por calificación (grade)')
ax.set_xlabel('Grade'); ax.set_ylabel('Tasa de mora (%)')
ax.grid(alpha=.3)
plt.show()

## 3. Principales factores de riesgo

Information Value (IV) por variable; umbral típico: IV ≥ 0.10 fuerte, 0.03–0.10 moderado, 0.02–0.03 débil.

In [ ]:
from src import scorecard

dev = m[m['issue_d'] < pd.Timestamp('2013-06-01')].copy()
ivs = {}
for f in features.ORIGINATION_FEATURES:
    s = dev[f]
    if s.dtype.kind == 'O' or s.nunique() <= 8:
        tbl = scorecard._woe_cat(dev['bad'].values, s, sorted(s.dropna().unique()))
    else:
        bins = scorecard.quantile_bins(s, n_bins=8)
        tbl = scorecard._woe_num(dev['bad'].values, s, bins)
    ivs[f] = tbl['iv'].sum()

iv_df = pd.DataFrame([{'variable': k, 'IV': v} for k, v in ivs.items()]).sort_values('IV', ascending=False)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(iv_df['variable'][::-1], iv_df['IV'][::-1], color='#2ca02c')
ax.set_title('Information Value por variable (muestra de desarrollo)');
ax.set_xlabel('IV');
plt.show()

## 4. Conclusión del EDA

1. El proxy Bueno/Malo construido discrimina de forma monotónica con la calificación interna.  
2. `grade`, `int_rate`, `term_months`, `income_per_loan` y `revol_util` concentran la información de riesgo.  
3. ~71% de la base está censurada (activa): el desarrollo debe hacerse sobre los préstamos resueltos.  
4. La tasa de mora de desarrollo es ≈ 12.1% (vintages 2007–2013).